# Lesson 10 · One number that measures "how alike" — dot product, cosine similarity, and projection

**Learning goals**
1. Compute a dot product by hand, and understand `a·b = |a||b|cosθ`
2. Implement `cosine_similarity(a, b)` (how alike the directions are, regardless of length)
3. Compute an orthogonal projection: how long the shadow is, and what the projection vector looks like

**Why it matters for Aurora**: each frequency bin `X[k]` in `dft()` is the dot product of the signal with a complex exponential sequence.
(The QK scoring in attention mechanisms later belongs to the same family — we unpack it in L83; this lesson nails down the dot product itself first.)

← **Previous lesson** · [L09 · Vector algebra](L09_vectors.en.ipynb)

> Last lesson covered **vector algebra**: addition, scalar multiplication and linear combination, with NumPy implementations + geometric meaning.
> This lesson explores **dot product and projection**.

## This lesson's story: the math behind music recommendations

How does Spotify know you'll like the next song?

It describes every song as a vector (genre, tempo, energy, ...), then finds the song whose **direction is closest** to the songs you've listened to. How "close" two directions are is measured with the **dot product**.

The dot product `a·b = Σaᵢbᵢ` squeezes two vectors into a single number:
- **Positive** → the two vectors point roughly the same way (you like electronic music, it recommends something similar)
- **Zero** → exactly perpendicular (the two songs have nothing in common)
- **Negative** → opposite directions (you like classical, it served you heavy metal)

But here's the catch: a 10-minute song has larger energy values — even if the style is very different, the dot product can still come out high.

The fix: divide by each song's own "energy size" (its norm), **normalizing** the dot product into [-1, 1]. That's **cosine similarity** — it doesn't care about vector length, only about direction.

In this lesson you'll implement `cosine_similarity(a, b)` and understand orthogonal projection (projection = dot product / magnitude).

## 1. Definition of the dot product

For two vectors `a = [a₁, a₂, ..., aₙ]` and `b = [b₁, b₂, ..., bₙ]`, the dot product is:

```
a · b = a₁b₁ + a₂b₂ + ... + aₙbₙ = Σᵢ aᵢbᵢ
```

A worked example by hand: a = [1, 2, 3], b = [4, 5, 6]

```
1×4 + 2×5 + 3×6 = 4 + 10 + 18 = 32
```

Geometric reading: `a·b = |a| · |b| · cos(θ)`, where θ is the angle between the two vectors.

This gives a second way to understand the dot product: **|a| · |b| · cos(θ) = the length-normalized "directional agreement" multiplied by the two lengths**.

In the DFT, `X[k] = Σ x[n] · e^{-2πikn/N}` is exactly the dot product of the signal x with the k-th twiddle factor — the "score" of each frequency component.

> 💡 `e^{-2πikn/N}` is a complex number. How the dot product deals with complex numbers gets its own explanation with a concrete example later in this lesson (after the parameter experiment) — for now just remember: it's the same "multiply element by element, then sum" logic as above, only one of the vectors is now complex.

## Digging deeper: why does a·b equal |a|·|b|·cos(θ)? — A tug-of-war story

Imagine two ropes pulled out from the same point at your feet, one pointing a little north of east (vector a), the other a little south of east (vector b). There's an angle θ between the two ropes. You want to know: how much are these two ropes "pulling in the same direction"?

If you just stare at the algorithmic definition "dot product = multiply matching positions, then add," you can't see what it has to do with "angle" — and that's exactly where many students get stuck. We'll start from **distance** and build a bridge.

**Step 1: draw the third side.** Draw the arrows a and b both from the origin. Their "difference" a − b is also a vector, and it's exactly the side connecting the two arrow tips; together with a and b it forms a triangle, with θ as the angle between a and b.

**Step 2: compute that side's squared length |a−b|² in two ways.**

Method A — pure algebra (expand component by component; works in any dimension):
```
|a - b|² = Σᵢ (aᵢ - bᵢ)²
         = Σᵢ aᵢ² - 2Σᵢ aᵢbᵢ + Σᵢ bᵢ²
         = |a|² + |b|² - 2 (a·b)
```
This step uses no geometry at all — it's purely expanding brackets and collecting like terms.

Method B — a geometry theorem (the relationship between sides and angles in a triangle):
The **law of cosines** from school tells us that the square of one side of a triangle = the sum of the squares of the other two sides − 2 × the product of those two sides × the cosine of the angle between them:
```
|a - b|² = |a|² + |b|² - 2 |a||b| cos(θ)
```

**Step 3: both methods compute the same number, so we can set them equal.**
```
|a|² + |b|² - 2(a·b)  =  |a|² + |b|² - 2|a||b|cos(θ)
```
Subtract `|a|² + |b|²` from both sides, then divide by −2:
```
a·b = |a| · |b| · cos(θ)
```

That's the bridge: **multiply element-wise and sum** (left side, the algebraic definition) and **length × length × cosine of the angle** (right side, the geometric definition) compute the same thing, just seen from two different angles — so when the vectors are perpendicular, cos(θ)=0 and the dot product is naturally 0 too. That's no coincidence; it's two ways of writing the same equation.

**Why cos and not sin?** Because this derivation goes through "side lengths" (distance), and the law of cosines is precisely the theorem that links "side lengths" to "angles"; sin shows up in formulas about "area" (like the area of a triangle, or the cross product), which measure "the surface spanned by two sides" — not what we're computing here.

**Do 1000-dimensional vectors still have an "angle"?** Yes — notice that Method A in Step 2 needs no picture at all: `Σ(aᵢ-bᵢ)²` gives you a number directly in 3 dimensions or 1000, with no need to "see" the angle. Since Methods A and B compute the same number, we can **run the equation backwards and use it to define** the angle between high-dimensional vectors:
```
cos(θ) := (a·b) / (|a| · |b|)
```
θ is no longer "an angle you see with your eyes" but "an angle computed from this formula" — which is also where the name "cosine similarity" comes from. Below we run this derivation on a concrete example.

In [1]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
from aurora.laviz import style, arrows2d

va = np.array([4.0, 1.0])
vb = np.array([1.0, 3.0])

# Method A: pure algebraic expansion, compute |a-b|² component by component
lhs = np.sum((va - vb) ** 2)

# Method B: first recover the angle theta from the dot product, then plug into the law of cosines
cos_theta = np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb))
theta = np.arccos(cos_theta)
rhs = (np.linalg.norm(va) ** 2 + np.linalg.norm(vb) ** 2
       - 2 * np.linalg.norm(va) * np.linalg.norm(vb) * np.cos(theta))

print(f"a = {va}, b = {vb}")
print(f"Method A |a-b|² (pure algebraic expansion) = {lhs:.4f}")
print(f"Method B |a-b|² (law of cosines)           = {rhs:.4f}")
assert np.isclose(lhs, rhs), "the two methods should give the same number"
print("The two methods agree ✅ — this is exactly why a·b = |a||b|cos(θ) holds")

style()
arrows2d([va, vb, va - vb], ['a', 'b', 'a - b'],
         title='The triangle formed by a, b and their difference a-b')

## Notation first: look at the shape, then the operation

Both inputs of a dot product must be vectors of the same dimension `(n,)`, and the output is a scalar. Every time you meet a new operation, first confirm that the input shapes match, then write the computation.

In [ ]:
import numpy as np
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print('element-wise product:', a * b)        # [4 10 18]
print('dot product (sum):', np.dot(a, b)) # 32
print('can also be written a @ b =', a @ b)

## Hands-on: two vectors, three angle relationships

Run the code below and watch how the sign of `np.dot(a, b)` changes with the angle between the two vectors: positive when they point the same way, zero when perpendicular, negative when opposite. That's the intuitive meaning of the dot product as a measure of "alignment."

In [ ]:
import numpy as np

# Three angle relationships: same direction, orthogonal, opposite
a = np.array([1., 0.])
cases = [
    ("same direction  cosθ = +1", np.array([ 2.,  0.])),
    ("orthogonal      cosθ =  0", np.array([ 0.,  1.])),
    ("opposite        cosθ = -1", np.array([-1.,  0.])),
]
print(f"fixed vector a = {a}")
for label, b in cases:
    dot = np.dot(a, b)
    cos = dot / (np.linalg.norm(a) * np.linalg.norm(b))
    print(f"  {label}:  a·b = {dot:+.2f},  cosine = {cos:+.2f}")


> **📌 L12 preview**: the cell below demonstrates a matrix scaling a vector. It's preview material from L12 (matrix transformations) — for now just look at the shapes; L12 derives it fully.


In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('vector length ||v|| =', np.linalg.norm(v))


## Code experiment: loop over a few vector pairs and watch how cosine similarity responds to direction

Compute the cosine similarity of several vector pairs in turn: identical → +1, orthogonal → 0, acute angle → close to +1. Notice that the result is decided only by **direction** and has nothing to do with vector length — that's exactly why cosine similarity normalizes the dot product.

In [ ]:
import numpy as np

# cosine similarity = dot product ÷ product of the two vectors' magnitudes
pairs = [
    (np.array([1.,0.]), np.array([1.,0.])),   # identical
    (np.array([1.,0.]), np.array([0.,1.])),   # orthogonal
    (np.array([1.,1.]), np.array([1.,-1.])),  # orthogonal
    (np.array([3.,4.]), np.array([4.,3.])),   # acute angle
]
for a, b in pairs:
    cos = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    print(f'cos({a}, {b}) = {cos:+.4f}')


## 2. Cosine similarity: how well do the directions agree (-1 to 1, closer to 1 = more alike)

`cos = (a·b) / (|a| · |b|)`, where `|a|` is the vector's length (covered in detail next lesson).

### A quick aside: what exactly is `|a|`? (No need to wait for next lesson)

A new symbol `|a|` just popped up in the formula above. Many textbooks also write it as `‖a‖` (double bars), and in code it corresponds to `np.linalg.norm(a)` — **all three notations mean the same thing**: the "length" of a vector, formally called its **norm**.

Why is length "sum of squares, then square root," rather than just adding up the components? Think about measuring the straight-line distance between two points on a map: if they're 3 squares apart horizontally and 4 vertically, the straight-line distance isn't 3+4=7 but `√(3²+4²) = 5`, from the Pythagorean theorem. A vector's length generalizes exactly the same idea — just extended from 2 components (horizontal and vertical) to n components:

```
|a| = ‖a‖ = np.linalg.norm(a) = √(a₁² + a₂² + ... + aₙ²)
```

For example `a = [1, 2, 3]`: `|a| = √(1² + 2² + 3²) = √14 ≈ 3.742`. Simply adding them up, `1+2+3=6`, is wrong because components can be positive or negative (e.g. a=[3,-4] should have length 5, not 3+(-4)=-1). Squaring guarantees every term is non-negative, and taking the square root brings you back to the units of "length."

This lesson just uses `|a|`; L11 covers its full definition (L1, L2, ∞ norms) and mathematical properties. For now just remember: **three notations, one meaning, and the recipe is "sum of squares, then square root"**.

In [ ]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])

manual_norm = np.sqrt(np.sum(a ** 2))   # square each component, sum, then take the square root
builtin_norm = np.linalg.norm(a)         # NumPy's built-in function computes the same thing

print(f"computed by hand √(1²+2²+3²) = {manual_norm:.4f}")
print(f"np.linalg.norm(a)            = {builtin_norm:.4f}")
assert np.isclose(manual_norm, builtin_norm)
print("They agree ✅ — |a| = ‖a‖ = np.linalg.norm(a) all mean the same thing")

## 3. ✏️ Your task: implement `cosine_similarity`

**Line of reasoning**:
1. The formula `cos = (a·b) / (|a|·|b|)` divides the dot product by the product of the two vectors' lengths, cancelling out size and leaving only direction — however long the vectors are, same-direction vectors always give 1.
2. The numerator is the dot product `Σ aᵢbᵢ`; compute it directly with `np.dot(a, b)`.
3. The denominator `|a|·|b|` is the product of two L2 norms (derived next lesson); for now use `np.linalg.norm()`.
4. Edge-case checks: same-direction vectors → 1; perpendicular vectors → 0; opposite vectors → -1.

**Reference input and output**: `a=[1,2,3]`, `b=[4,5,6]` → dot product = 32, |a|=√14≈3.742, |b|=√77≈8.775, cosine ≈ **0.974**

> 💡 Need a hint? Once you've finished the exercise, you can check the reference implementation in the `solutions/` folder.


### Three things to pin down before writing `cosine_similarity`

- Input: `a`, `b`, both of shape `(n,)`, and the dimensions must match
- Key step: numerator `np.dot(a, b)`, denominator `np.linalg.norm(a) * np.linalg.norm(b)`
- Returns: a scalar in the range [-1, 1]; perpendicular vectors should give 0, same-direction vectors 1

### A quick aside: why does the zero vector get its own error?

The code requires: "if either vector is the zero vector (magnitude 0), raise ValueError." This isn't to make life hard for you — mathematically there really is no result to compute.

The formula is `cos = (a·b) / (|a| · |b|)`. If `a` is the zero vector `[0, 0, ..., 0]`, then `|a| = 0` and the denominator becomes 0 — and "dividing by 0" is undefined in mathematics (not "infinity," but "no answer"). When Python/NumPy meets `0.0 / 0.0` it doesn't raise an error right away; it quietly hands you a `nan` (not a number). If you don't stop it early, the program carries on with this "bad number," and the place where things actually break ends up far downstream, where it's very hard to track down.

**Do zero vectors really show up in practice?** They do. For example, a brand-new song that nobody has listened to yet and that hasn't been tagged with any style labels may have an all-zero feature vector (in recommender systems this is called the "cold start" problem); or an upstream data pipeline breaks and accidentally wipes a song's features to 0. In those cases the question "which songs is this one most like?" is meaningless in itself — **no direction, no "alike or not."** Rather than letting the program quietly compute a `nan` and crash dozens of lines later, it's better to raise an error right now and expose the problem at its source. The code below shows what happens if you skip the check:

In [ ]:
import numpy as np

zero = np.array([0.0, 0.0, 0.0, 0.0])
song = np.array([5.0, 0.0, 3.0, 0.0])

with np.errstate(invalid='ignore', divide='ignore'):
    silent_result = np.dot(zero, song) / (np.linalg.norm(zero) * np.linalg.norm(song))
print(f"Without a zero-vector check, the computation quietly gives: {silent_result}")
print("← This is what happens when you don't intercept the zero vector: the nan travels all the way down and only blows up somewhere else much later")

In [ ]:
def cosine_similarity(a, b):
    """
    Return the cosine similarity of vectors a and b, in the range [-1, 1].
    If either vector is the zero vector (magnitude 0), raise ValueError.
    """
    # ✏️ TODO: implement cosine similarity
    # Hint: numerator np.dot(a, b), denominator np.linalg.norm(a) * np.linalg.norm(b)
    # Zero-vector handling: if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0: raise ValueError("...")
    raise NotImplementedError("Please implement cosine_similarity")


In [ ]:
song1 = np.array([5.0, 0.0, 3.0, 0.0])  # pretend this is a song's feature vector
song2 = np.array([4.0, 0.0, 2.0, 0.0])  # very similar to song1
song3 = np.array([0.0, 5.0, 0.0, 4.0])  # a different style

try:
    print("sim(1,2) =", round(cosine_similarity(song1, song2), 3), "(should be close to 1, very alike)")
    print("sim(1,3) =", round(cosine_similarity(song1, song3), 3), "(should be close to 0, not alike)")
    assert abs(cosine_similarity(song1, 2 * song1) - 1.0) < 1e-12, "cosine similarity of same-direction vectors should be exactly 1"
    assert cosine_similarity(song1, song2) > 0.9, "similarity of songs of the same kind should be > 0.9"
    assert abs(cosine_similarity(song1, song3)) < 1e-10, "song1⊥song3, similarity should be exactly 0"
    assert cosine_similarity(song1, song2) > cosine_similarity(song1, song3)
    # Edge case: the zero vector should trigger ValueError
    try:
        cosine_similarity(np.zeros(4), song1)
        assert False, "the zero vector should trigger ValueError"
    except ValueError:
        print("✅ Zero-vector edge case: ValueError raised correctly")
    print("\n✅ All passed: you've built the core metric of a recommender system.")
except (NotImplementedError, TypeError) as e:
    print(f"⚠️  Not implemented yet: {e}\nPlease fill in the body of cosine_similarity in the cell above.")


**🔗 Aurora connection**: the Music Core's "you might also like" = find the song with the highest cosine similarity. In the DFT, `X[k]` = signal · the complex exponential at the k-th frequency, which is also a dot product.

## 🎨 Diagram: the outer product (→ a rank-1 matrix)

Inner product a·b = a scalar (printed below); outer product a⊗b = a rank-1 matrix (heatmap below).

In [ ]:
import numpy as np
from aurora.laviz import style, vec_times_vec
style()
a_vis, b_vis = [1, 2, 3], [4, 5, 6]
inner = int(sum(x*y for x, y in zip(a_vis, b_vis)))
print(f"inner product a·b = {inner}  (a scalar: squeezed into one number)")
print("outer product a⊗b = a rank-1 matrix (expanded into a matrix, see the diagram):")
vec_times_vec(a_vis, b_vis);


### Digging deeper: why are they called the "inner" and "outer" product? And what is a "rank-1 matrix"?

**Why is one "inner" and the other "outer"?** Think of them as two actions going in opposite directions:

- **Inner product** (the dot product is one kind of inner product): two vectors contract "inward" and get squeezed into a single number — two pieces of information become one; the dimensions fold inward.
- **Outer product**: two vectors unfold "outward" and spread into a whole matrix — one piece of information multiplies out into a whole grid; the dimensions expand outward.

One "contracts," the other "expands" — exactly opposite directions. That's where the words "inner" and "outer" come from.

**What does "rank-1 matrix" mean?** Roughly speaking, the "rank" of a matrix is the number of "independent directions" in it — how many directions across the matrix's rows (or columns) actually bring new information. Look at the matrix behind the heatmap below: each row of the outer product `a⊗b` is just the vector `b` times a different factor (row 1 is `a₁×b`, row 2 is `a₂×b`, …). Every row is talking about "the direction of b," only with different factors — the whole matrix really holds information about just "one direction," which is why an outer product is always a **rank-1** matrix, however many dimensions a and b have. Matrix rank gets a full definition in L12 (matrix transformations); for now, build this intuition: **rank 1 = every row of the matrix is just a different multiple of the same vector**. Let's verify:

In [ ]:
import numpy as np

a_vis, b_vis = np.array([1, 2, 3]), np.array([4, 5, 6])
outer = np.outer(a_vis, b_vis)
print("outer product matrix a⊗b =")
print(outer)
print("\nIs every row a multiple of b?")
for i, row in enumerate(outer):
    print(f"  row {i+1}: {row} = {a_vis[i]} × {b_vis}")
print(f"\nnp.linalg.matrix_rank(outer) = {np.linalg.matrix_rank(outer)}  ← confirms it really is rank 1")

> **📌 L12 preview**: the matrix-probe experiment below shows what a linear transformation does to several vectors; the full story is in L12.


In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('How does matrix A move these vectors?')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | length {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## Parameter experiment: turn one knob at a time

Take `t = np.linspace(0, 1, 1000)`, and let `a = np.sin(2*np.pi*1*t)` (1 Hz) and `b = np.sin(2*np.pi*2*t)` (2 Hz) — different frequencies. Compute `np.dot(a, b)`: the result is close to 0, because sine waves of different frequencies are orthogonal to each other. Now let `b = np.sin(2*np.pi*1*t)` (same frequency) and the dot product becomes positive; let `b = 2 * np.sin(2*np.pi*1*t)` and the dot product doubles — the dot product is proportional to the product of the amplitudes. The DFT's `X[k]` exploits exactly this property: take the dot product with the complex exponential at the k-th frequency, and the matching frequency gives the largest magnitude while the other frequencies cancel out thanks to orthogonality.

## Digging deeper: what is `e^{-2πikn/N}` in the DFT? How does the dot product deal with complex numbers?

The experiment above only took dot products of real signals (sine waves), and the result is already revealing: same frequency → large dot product, different frequencies → dot product near 0. The "detector" formula the DFT actually uses is:

```
X[k] = Σₙ x[n] · e^{-2πikn/N}
```

`e^{-2πikn/N}` looks scary, but take it apart and it's an old friend — Euler's formula from L06:
```
e^{-iθ} = cos(θ) - i·sin(θ), where θ = 2πkn/N
```
In other words, `e^{-2πikn/N}` is a complex number that **packs both a cos detector and a sin detector at once** — the real part matches against the cosine waveform, the imaginary part against the sine waveform, so one dot product does the work of two. That's also why the DFT needs just one line of formula to capture two pieces of information from a signal at once: "how much does it look like the cos waveform" and "how much does it look like the sin waveform."

**So how do you take a "dot product" of complex vectors?** The ordinary dot product `Σaᵢbᵢ` runs into a small snag with complex numbers: if a and b are both complex, `a·a` isn't necessarily positive (it may not even be real), so it can't serve as "the squared length." To keep the concept of "length" working in the complex world, the complex dot product (strictly speaking, the **inner product**) adds one step to one of the vectors: **taking the conjugate** (flipping the sign of the imaginary part, `i` becomes `-i`):
```
complex dot product: a·b = Σᵢ aᵢ · conj(bᵢ)
```
If x[n] is a real signal (its imaginary part is 0 to begin with), conjugation has no effect on it, so the "dot product" in the DFT formula is, for a real signal x, just honest element-wise multiplication and summation — it's only "the other vector" (the complex exponential sequence) that is complex.

Below, a mini example with just 3 samples walks through computing `X[1]` step by step, turning the formula into concrete numbers:

In [ ]:
import numpy as np

N = 3
k = 1
x = np.array([1.0, 2.0, 3.0])   # a "signal" (real-valued) with just 3 points

# The k=1 complex exponential sequence (twiddle factors): e^(-2πi·k·n/N), n = 0, 1, 2
n = np.arange(N)
w = np.exp(-2j * np.pi * k * n / N)
print(f"complex exponential sequence w = e^(-2πi·{k}n/{N}) =")
for i, wi in enumerate(w):
    print(f"  n={i}: {wi:.4f}   (real part≈cos, imaginary part≈-sin)")

# Multiply term by term and sum by hand — x is real, so conjugation doesn't change it; just use w's values
X_k_manual = sum(x[i] * w[i] for i in range(N))
X_k_numpy = np.dot(x, w)   # when x is real, this dot product is equivalent to element-wise multiply-and-sum

print(f"\naccumulated by hand X[{k}] = Σ x[n]·w[n] = {X_k_manual:.4f}")
print(f"np.dot(x, w)                    = {X_k_numpy:.4f}")
assert np.isclose(X_k_manual, X_k_numpy)
print("\n✅ For a real signal, the dot product is just element-wise multiply-and-sum — only one of the 'vectors' is complex")

## 4. Orthogonal projection: a geometric application of the dot product

**Scalar projection** (the length of a's projection onto the direction of b):

$$\text{scalar\_proj}_b(a) = \frac{a \cdot b}{\lvert b \rvert}$$

**Vector projection** (the projection vector of a onto the direction of b):

$$\text{vec\_proj}_b(a) = \frac{a \cdot b}{\lvert b \rvert^2} \cdot b$$

### Where do these two formulas come from? — From "how long is the shadow" to "what does the shadow look like"

First imagine the sun directly overhead, b as the direction of a pole lying on the ground, and a as another pole standing at a slant. For the shadow that a casts on the ground (that is, along b), you can ask two questions:

1. **How long is the shadow?** — That's the "scalar projection," a single number.
2. **What exactly is the shadow as a vector?** (which direction and how long, all at once) — That's the "vector projection."

**Step 1: first work out "how long the shadow is."** The length of a projected onto the direction of b is a's component along b. Divide the dot product by b's length:
```
scalar_proj = a·b / |b|
```
(Where the intuition comes from: we derived `a·b = |a||b|cos(θ)` above; divide both sides by `|b|` and you're left with `|a|cos(θ)` — exactly "hypotenuse × cosine of the angle = length of the adjacent side" in a right triangle, i.e. the length of the shadow.)

**Step 2: the shadow has not only a length but also a direction — the direction of b.** To turn a "length" into a "vector," just multiply by a vector **of length 1 pointing along b**, i.e. b's **unit vector** `b / |b|`:
```
vec_proj = scalar_proj × (unit vector along b)
         = (a·b / |b|) × (b / |b|)
         = (a·b) × b / |b|²
```
Two factors of `|b|` multiply to make the denominator `|b|²` — and that's the whole reason "the scalar projection has `|b|` in the denominator but the vector projection has `|b|²`": **the extra `|b|` you divide by is there to turn b into a unit vector first, so that it carries the direction.**

In code, `np.dot(b, b)` computes exactly `|b|²` (because `b·b = Σbᵢ×bᵢ = Σbᵢ² = |b|²` — a handy little identity worth remembering; it saves a lot of effort).

Geometric meaning: cast a's "shadow" onto the direction of b.
- a ⊥ b (orthogonal) → the projection is the zero vector
- a ∥ b (same direction) → the projection is a itself
- The heart of the least squares method: finding the best-fit line is equivalent to projecting so that the error vector is as small as possible

### Digging deeper: what does "projecting so that the error vector is as small as possible" actually mean?

The phrase feels like a leap because it crams two actions into one: first "build an error vector," then "figure out how to make it smallest." Let's take it apart:

Suppose you have only 3 data points and want to find a line through the origin `y = c·x` that fits them (we tune just 1 parameter c, to keep the intuition simple). For each candidate c, there's always some gap between the "predicted values" and the "true values"; stack all the gaps together and you get the **error vector** `e = y_true − c·x`.

The c that least squares is looking for is the one that makes the length `|e|` of this error vector smallest. The code below actually tries a few values of c, and you'll see: **when |e| is smallest, the error vector e is exactly perpendicular to x** (`e·x ≈ 0`) — in other words, the optimal solution lands exactly at "projecting y onto the direction of x." That's the picture behind the phrase "least squares = projection."

**Aurora connection**: the spectral analysis in `src/aurora/audio/` projects the signal onto each sine/cosine basis function — the projection interpretation of the DFT.


In [ ]:
import numpy as np

def scalar_projection(a, b):
    """Scalar projection of a onto the direction of b = a·b / |b|"""
    return np.dot(a, b) / np.linalg.norm(b)

def vector_projection(a, b):
    """Vector projection of a onto the direction of b = (a·b / |b|²) * b"""
    return (np.dot(a, b) / np.dot(b, b)) * b

# Example: projecting a=[3,4] onto the x-axis direction b=[1,0]
a = np.array([3., 4.])
b = np.array([1., 0.])
print(f"a = {a}, b = {b}")
print(f"scalar projection (length along b) = {scalar_projection(a, b):.4f}")  # should be 3.0
print(f"vector projection (vector along b) = {vector_projection(a, b)}")       # should be [3. 0.]

# Check: projecting onto an orthogonal vector gives zero (c is orthogonal to a=[3,4]: 3×4 + 4×(-3) = 0)
c = np.array([4., -3.])
print(f"\nwhen a ⊥ c, dot product = {np.dot(a, c)}")
print(f"vector projection of a onto c = {vector_projection(a, c)}")  # should be [0. 0.]

assert abs(scalar_projection(a, b) - 3.0) < 1e-10, "scalar projection should be 3.0"
assert np.allclose(vector_projection(a, b), [3., 0.]), "vector projection should be [3, 0]"
assert np.allclose(vector_projection(a, c), [0., 0.]), "projection onto an orthogonal vector should be the zero vector"
print("\n✅ Projection checks passed")

In [ ]:
import numpy as np

# 3 data points; fit a line through the origin y = c * x with a single parameter c
x = np.array([1.0, 2.0, 3.0])
y = np.array([2.2, 3.8, 6.1])   # true observations (with a little noise)

print("Try a few candidate slopes c and look at the length of the error vector:")
for c in [1.5, 1.8, 2.0, 2.05, 2.2]:
    e = y - c * x                       # error vector
    print(f"  c={c:.2f}  |e|={np.linalg.norm(e):.4f}  e·x={np.dot(e, x):+.4f}")

# The closed-form least-squares solution is exactly the ratio from projecting y onto the direction of x
c_star = np.dot(x, y) / np.dot(x, x)    # the same ratio scalar_projection uses
e_star = y - c_star * x
print(f"\noptimal c* = (x·y)/(x·x) = {c_star:.4f}")
print(f"error vector at this point e* = {e_star}")
print(f"e*·x = {np.dot(e_star, x):.6f}  ← close to 0: the error is perpendicular to x (the projection is optimal)")

## Wrap-up

`cosine_similarity(a, b)` can now normalize `Σaᵢbᵢ` into [-1, 1]. Scalar/vector projections are the geometric output of the dot product — each DFT frequency bin is also the projection of the signal onto the corresponding basis. In Aurora's `dft()`, computing each `X[k]` is the dot product of the signal array with a complex exponential sequence, with exactly the same algebraic structure as in this lesson. Next lesson: **L11** covers norms — the source of the denominator in cosine similarity, and the tool for measuring a vector's "size."

## ✏️ Whiteboard challenge: dot product and cosine similarity by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Question 1**: a = [1, 2, 3], b = [4, 5, 6]; compute `a·b` by hand (multiply element-wise, then sum).

**Question 2**: `cosine_similarity([1, 0], [0, 1])` = ? (the two vectors are perpendicular — no calculator)

**Question 3**: `cosine_similarity([3, 0], [6, 0])` = ? (same direction, different lengths)

**Question 4**: `scalar_projection([3, 4], [1, 0])` = ? (projecting the vector [3,4] onto the x-axis)
Formula: `a·b / |b|`, where b = [1, 0].

When you've finished, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: dot product
a, b = np.array([1.0, 2.0, 3.0]), np.array([4.0, 5.0, 6.0])
q1 = float(np.dot(a, b))
assert q1 == 32.0, f"expected 32, got {q1}"
print(f"Q1 ✅  [1,2,3]·[4,5,6] = 1×4 + 2×5 + 3×6 = {q1}")

# Q2: cosine similarity of perpendicular vectors
_pending = 0
try:
    q2 = cosine_similarity(np.array([1.0, 0.0]), np.array([0.0, 1.0]))
    assert np.isclose(q2, 0.0, atol=1e-10)
    print(f"Q2 ✅  cos([1,0], [0,1]) = {q2:.4f} (perpendicular → 0)")
except (NotImplementedError, TypeError):
    print("Q2 ⬜  cos(90°) = 0 (cosine_similarity not implemented yet)")
    _pending += 1

# Q3: same direction, different lengths
try:
    q3 = cosine_similarity(np.array([3.0, 0.0]), np.array([6.0, 0.0]))
    assert np.isclose(q3, 1.0, atol=1e-10)
    print(f"Q3 ✅  cos([3,0], [6,0]) = {q3:.4f} (same direction → 1.0, regardless of length)")
except (NotImplementedError, TypeError):
    print("Q3 ⬜  same direction → cos = 1.0 (cosine_similarity not implemented yet)")
    _pending += 1

# Q4: scalar projection
a4, b4 = np.array([3.0, 4.0]), np.array([1.0, 0.0])
q4 = float(np.dot(a4, b4)) / float(np.linalg.norm(b4))
assert np.isclose(q4, 3.0, atol=1e-10)
print(f"Q4 ✅  scalar_proj([3,4] → x-axis) = {q4} (just the x component)")
if _pending:
    print(f"\n⬜ {_pending} question(s) left — implement cosine_similarity, then check again.")
else:
    print("\n🎉 Dot-product whiteboard challenge passed! Cosine similarity = directional agreement, quantified.")

In [ ]:
# ✏️ Self-review for this lesson
l10_review = {
    "cosine_similarity_implemented": None,  # cosine_similarity implemented and passes its asserts? True/False
    "dot_product_geometry":          None,  # understand the geometric meaning of a·b = |a||b|cosθ? True/False
    "orthogonality_intuition":       None,  # know that dot product = 0 ↔ perpendicular? True/False
    "projection_understood":         None,  # can explain the scalar/vector projection formulas? True/False
    "whiteboard_passed":             None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l10_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l10_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L10 complete! On to L11: vector norms')

---

→ **Next lesson** · [L11 · Vector norms](L11_norms.ipynb) *(Chinese)*

> Next lesson covers **vector norms**: computing L1 / L2 / ∞ norms, their geometric shapes, and what they mean for regularization.